In [1]:
import pandas as pd
import json
from pathlib import Path

RAW_DIR = Path("/content")
PROCESSED_DIR = Path("/content/processed")
PYTHON_DIR = Path("/content/python_outputs")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PYTHON_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


In [25]:
ledger = pd.read_csv(RAW_DIR / "ledger.csv")
gateway = pd.read_csv(RAW_DIR / "gateway.csv")

print("Ledger shape:", ledger.shape)
print("Gateway shape:", gateway.shape)

display(ledger.head())
display(gateway.head())

Ledger shape: (10, 6)
Gateway shape: (9, 6)


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,01-03-2026,M001,1200,success,UPI
1,R002,01-03-2026,M002,850,success,Card
2,R003,02-03-2026,M001,500,success,Wallet
3,R004,02-03-2026,M003,2100,success,Card
4,R005,03-03-2026,M004,7200,success,Card


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,01-03-2026,M001,1200,success,UPI
1,R002,01-03-2026,M002,900,success,Card
2,R003,02-03-2026,M001,500,success,Wallet
3,R005,03-03-2026,M004,7200,failed,Card
4,R006,03-03-2026,M002,950,success,UPI


In [3]:
missing_in_gateway = ledger[
    ~ledger["transaction_id"].isin(gateway["transaction_id"])
]

print("Missing in gateway:")
display(missing_in_gateway)

missing_in_gateway.to_csv(
    PROCESSED_DIR / "missing_in_gateway.csv",
    index=False
)

Missing in gateway:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
3,R004,02-03-2026,M003,2100,success,Card
9,R010,05-03-2026,M004,2500,success,Wallet


In [4]:
missing_in_ledger = gateway[
    ~gateway["transaction_id"].isin(ledger["transaction_id"])
]

print("Missing in ledger:")
display(missing_in_ledger)

missing_in_ledger.to_csv(
    PROCESSED_DIR / "missing_in_ledger.csv",
    index=False
)

Missing in ledger:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
8,R011,05-03-2026,M003,1800,success,Card


In [5]:
merged = pd.merge(
    ledger,
    gateway,
    on="transaction_id",
    how="inner",
    suffixes=("_ledger", "_gateway")
)

print("Merged shape:", merged.shape)

display(merged.head())

Merged shape: (8, 11)


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
0,R001,01-03-2026,M001,1200,success,UPI,01-03-2026,M001,1200,success,UPI
1,R002,01-03-2026,M002,850,success,Card,01-03-2026,M002,900,success,Card
2,R003,02-03-2026,M001,500,success,Wallet,02-03-2026,M001,500,success,Wallet
3,R005,03-03-2026,M004,7200,success,Card,03-03-2026,M004,7200,failed,Card
4,R006,03-03-2026,M002,950,success,UPI,03-03-2026,M002,950,success,UPI


In [6]:
amount_mismatches = merged[
    merged["amount_usd_ledger"] != merged["amount_usd_gateway"]
]

print("Amount mismatches:")
display(amount_mismatches)

amount_mismatches.to_csv(
    PROCESSED_DIR / "amount_mismatches.csv",
    index=False
)

Amount mismatches:


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
1,R002,01-03-2026,M002,850,success,Card,01-03-2026,M002,900,success,Card
6,R008,04-03-2026,M001,640,success,Card,04-03-2026,M001,600,success,Card


In [7]:
status_mismatches = merged[
    merged["status_ledger"].str.lower().str.strip()
    != merged["status_gateway"].str.lower().str.strip()
]

print("Status mismatches:")
display(status_mismatches)

status_mismatches.to_csv(
    PROCESSED_DIR / "status_mismatches.csv",
    index=False
)

Status mismatches:


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
3,R005,03-03-2026,M004,7200,success,Card,03-03-2026,M004,7200,failed,Card


In [8]:
# Start with all transaction IDs from both systems
all_transaction_ids = pd.DataFrame({
    "transaction_id": sorted(
        set(ledger["transaction_id"]).union(set(gateway["transaction_id"]))
    )
})

# Build issue flags
all_transaction_ids["missing_in_gateway"] = all_transaction_ids["transaction_id"].isin(
    missing_in_gateway["transaction_id"]
).astype(int)

all_transaction_ids["missing_in_ledger"] = all_transaction_ids["transaction_id"].isin(
    missing_in_ledger["transaction_id"]
).astype(int)

all_transaction_ids["amount_mismatch"] = all_transaction_ids["transaction_id"].isin(
    amount_mismatches["transaction_id"]
).astype(int)

all_transaction_ids["status_mismatch"] = all_transaction_ids["transaction_id"].isin(
    status_mismatches["transaction_id"]
).astype(int)

# Final issue label
def classify_issue(row):
    issues = []
    if row["missing_in_gateway"] == 1:
        issues.append("missing_in_gateway")
    if row["missing_in_ledger"] == 1:
        issues.append("missing_in_ledger")
    if row["amount_mismatch"] == 1:
        issues.append("amount_mismatch")
    if row["status_mismatch"] == 1:
        issues.append("status_mismatch")
    return ", ".join(issues) if issues else "matched"

all_transaction_ids["reconciliation_status"] = all_transaction_ids.apply(classify_issue, axis=1)

reconciliation_report = all_transaction_ids

display(reconciliation_report)

reconciliation_report.to_csv(
    PROCESSED_DIR / "reconciliation_report.csv",
    index=False
)

,transaction_id,missing_in_gateway,missing_in_ledger,amount_mismatch,status_mismatch,reconciliation_status
0,R001,0,0,0,0,matched
1,R002,0,0,1,0,amount_mismatch
2,R003,0,0,0,0,matched
3,R004,1,0,0,0,missing_in_gateway
4,R005,0,0,0,1,status_mismatch
5,R006,0,0,0,0,matched
6,R007,0,0,0,0,matched
7,R008,0,0,1,0,amount_mismatch
8,R009,0,0,0,0,matched
9,R010,1,0,0,0,missing_in_gateway


In [9]:
# Save reconciliation outputs as actual CSV files

missing_in_gateway.to_csv("/content/processed/missing_in_gateway.csv", index=False)
missing_in_ledger.to_csv("/content/processed/missing_in_ledger.csv", index=False)
amount_mismatches.to_csv("/content/processed/amount_mismatches.csv", index=False)
status_mismatches.to_csv("/content/processed/status_mismatches.csv", index=False)
reconciliation_report.to_csv("/content/processed/reconciliation_report.csv", index=False)

print("CSV files saved successfully.")

CSV files saved successfully.


In [10]:
import os

for file in os.listdir("/content/processed"):
    print(file)

status_mismatches.csv
amount_mismatches.csv
missing_in_gateway.csv
reconciliation_report.csv
missing_in_ledger.csv


In [11]:
with open("/content/api_response_sample.json", "r") as f:
    api_data = json.load(f)

print(type(api_data))
print(api_data)

<class 'dict'>
{'generated_at': '2026-03-07T10:00:00Z', 'source': 'QuickPay Settlement API', 'batches': [{'batch_id': 'B001', 'merchant': {'merchant_id': 'M001', 'merchant_name': 'Alpha Mart', 'region': 'APAC'}, 'settlements': [{'settlement_id': 'S001', 'amount_usd': 1520.5, 'status': 'settled', 'processed_at': '2026-03-07T08:10:00Z', 'bank': {'name': 'Bank A', 'country': 'IN'}}, {'settlement_id': 'S002', 'amount_usd': 980.0, 'status': 'pending', 'processed_at': '2026-03-07T08:45:00Z', 'bank': {'name': 'Bank A', 'country': 'IN'}}, {'settlement_id': 'S003', 'amount_usd': 640.0, 'status': 'settled', 'processed_at': '2026-03-07T09:15:00Z', 'bank': {'name': 'Bank B', 'country': 'SG'}}]}, {'batch_id': 'B002', 'merchant': {'merchant_id': 'M004', 'merchant_name': 'Delta Travels', 'region': 'US'}, 'settlements': [{'settlement_id': 'S004', 'amount_usd': 2100.0, 'status': 'settled', 'processed_at': '2026-03-07T08:20:00Z', 'bank': {'name': 'Bank C', 'country': 'US'}}, {'settlement_id': 'S005', 'a

In [12]:
api_normalized = pd.json_normalize(
    api_data,
    record_path=["batches", "settlements"],
    meta=[
        "generated_at",
        "source",
        ["batches", "batch_id"],
        ["batches", "merchant", "merchant_id"],
        ["batches", "merchant", "merchant_name"],
        ["batches", "merchant", "region"]
    ],
    sep="_"
)

# clean column names
api_normalized.columns = (
    api_normalized.columns
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "_", regex=False)
)

# convert datetime fields
api_normalized["processed_at"] = pd.to_datetime(api_normalized["processed_at"])
api_normalized["generated_at"] = pd.to_datetime(api_normalized["generated_at"])

display(api_normalized)

api_normalized.to_csv(
    "/content/processed/api_normalized.csv",
    index=False
)

print("api_normalized.csv saved successfully.")

,settlement_id,amount_usd,status,processed_at,bank_name,bank_country,generated_at,source,batches_batch_id,batches_merchant_merchant_id,batches_merchant_merchant_name,batches_merchant_region
0,S001,1520.5,settled,2026-03-07 08:10:00+00:00,Bank A,IN,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B001,M001,Alpha Mart,APAC
1,S002,980.0,pending,2026-03-07 08:45:00+00:00,Bank A,IN,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B001,M001,Alpha Mart,APAC
2,S003,640.0,settled,2026-03-07 09:15:00+00:00,Bank B,SG,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B001,M001,Alpha Mart,APAC
3,S004,2100.0,settled,2026-03-07 08:20:00+00:00,Bank C,US,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B002,M004,Delta Travels,US
4,S005,500.0,failed,2026-03-07 08:50:00+00:00,Bank C,US,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B002,M004,Delta Travels,US
5,S006,7200.0,settled,2026-03-07 09:30:00+00:00,Bank C,US,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B002,M004,Delta Travels,US


api_normalized.csv saved successfully.


In [15]:
amount_at_risk = (
    missing_in_gateway["amount_usd"].sum()
    + missing_in_ledger["amount_usd"].sum()
    + amount_mismatches["amount_usd_ledger"].sum()
    + status_mismatches["amount_usd_ledger"].sum()
)

summary_metrics = {
    "total_ledger_rows": int(len(ledger)),
    "total_gateway_rows": int(len(gateway)),
    "missing_in_gateway_count": int(len(missing_in_gateway)),
    "missing_in_ledger_count": int(len(missing_in_ledger)),
    "amount_mismatch_count": int(len(amount_mismatches)),
    "status_mismatch_count": int(len(status_mismatches)),
    "reconciliation_issue_count": int(
        len(missing_in_gateway)
        + len(missing_in_ledger)
        + len(amount_mismatches)
        + len(status_mismatches)
    ),
    "ledger_total_amount": float(ledger["amount_usd"].sum()),
    "gateway_total_amount": float(gateway["amount_usd"].sum()),
    "amount_at_risk": float(amount_at_risk)
}

print(json.dumps(summary_metrics, indent=4))

with open("/content/summary_metrics.json", "w") as f:
    json.dump(summary_metrics, f, indent=4)

print("summary_metrics.json saved successfully.")

{
    "total_ledger_rows": 10,
    "total_gateway_rows": 9,
    "missing_in_gateway_count": 2,
    "missing_in_ledger_count": 1,
    "amount_mismatch_count": 2,
    "status_mismatch_count": 1,
    "reconciliation_issue_count": 6,
    "ledger_total_amount": 23340.0,
    "gateway_total_amount": 20550.0,
    "amount_at_risk": 15090.0
}
summary_metrics.json saved successfully.


In [18]:
transactions = pd.read_csv("/content/cleaned_transactions.csv")

print(transactions.shape)

display(transactions.head())

(30, 13)


,transaction_id,transaction_date,merchant_name,raw_amount,currency,status,risk_score,gateway_region,user_id,payment_method,amount_usd,high_value_flag,high_risk_flag
0,T001,01-03-2026,ALPHA MART,420000,INR,Captured,62.0,APAC,U001,UPI,4998.0,0,0
1,T002,01-03-2026,ALPHA MART,210000,INR,Captured,55.0,NaN,U002,Card,2499.0,0,0
2,T003,01-03-2026,BETA STORES,510000,INR,Captured,71.0,APAC,U003,NetBanking,6069.0,1,1
3,T004,02-03-2026,BETA STORES,160000,INR,Failed,68.0,APAC,U004,Card,1904.0,0,0
4,T005,02-03-2026,ALPHA MART,390000,INR,Captured,58.0,NaN,U001,UPI,4641.0,0,0


In [19]:
print(transactions.columns.tolist())

['transaction_id', 'transaction_date', 'merchant_name', 'raw_amount', 'currency', 'status', 'risk_score', 'gateway_region', 'user_id', 'payment_method', 'amount_usd', 'high_value_flag', 'high_risk_flag']


In [20]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"], errors="coerce")
transactions["status_clean"] = transactions["status"].str.lower().str.strip()

In [21]:
daily_summary = (
    transactions
    .groupby("transaction_date")
    .agg(
        total_gmv=("amount_usd", "sum"),
        confirmed_gmv=("amount_usd", lambda x: x[transactions.loc[x.index, "status_clean"] == "captured"].sum()),
        total_transactions=("transaction_id", "count"),
        successful_transactions=("status_clean", lambda x: (x == "captured").sum()),
        high_risk_transactions=("high_risk_flag", "sum")
    )
    .reset_index()
)

daily_summary["success_rate"] = (
    daily_summary["successful_transactions"] / daily_summary["total_transactions"]
).round(3)

daily_summary.to_csv("/content/processed/daily_summary.csv", index=False)

display(daily_summary)

,transaction_date,total_gmv,confirmed_gmv,total_transactions,successful_transactions,high_risk_transactions,success_rate
0,2026-01-03,26382.0,26382.0,5,5,1,1.000
1,2026-02-03,24860.5,11013.5,6,3,1,0.500
2,2026-03-03,18137.0,15816.5,5,4,2,0.800
3,2026-04-03,16304.0,13804.0,5,4,1,0.800
4,2026-05-03,19331.5,6188.0,6,1,3,0.167
5,2026-06-03,10606.0,8806.0,3,2,0,0.667


In [22]:
payment_method_breakdown = (
    transactions
    .groupby("payment_method")
    .agg(
        total_transactions=("transaction_id", "count"),
        total_gmv=("amount_usd", "sum"),
        confirmed_gmv=("amount_usd", lambda x: x[transactions.loc[x.index, "status_clean"] == "captured"].sum()),
        high_risk_transactions=("high_risk_flag", "sum")
    )
    .reset_index()
)

payment_method_breakdown.to_csv("/content/processed/payment_method_breakdown.csv", index=False)

display(payment_method_breakdown)

,payment_method,total_transactions,total_gmv,confirmed_gmv,high_risk_transactions
0,Card,13,52730.0,34123.0,3
1,NetBanking,3,15226.0,11662.0,2
2,UPI,9,34287.0,28441.0,3
3,Wallet,5,13378.0,7784.0,0


In [23]:
region_breakdown = (
    transactions
    .groupby("gateway_region")
    .agg(
        total_transactions=("transaction_id", "count"),
        total_gmv=("amount_usd", "sum"),
        confirmed_gmv=("amount_usd", lambda x: x[transactions.loc[x.index, "status_clean"] == "captured"].sum()),
        avg_risk_score=("risk_score", "mean"),
        high_risk_transactions=("high_risk_flag", "sum")
    )
    .reset_index()
)

region_breakdown["avg_risk_score"] = region_breakdown["avg_risk_score"].round(2)

region_breakdown.to_csv("/content/processed/region_breakdown.csv", index=False)

display(region_breakdown)

,gateway_region,total_transactions,total_gmv,confirmed_gmv,avg_risk_score,high_risk_transactions
0,APAC,13,50099.0,35878.5,67.54,6
1,EU,4,18792.0,8640.0,47.25,0
2,US,4,14600.0,10300.0,48.75,0


In [24]:
merchant_performance_summary = (
    transactions
    .groupby("merchant_name")
    .agg(
        total_transactions=("transaction_id", "count"),
        total_gmv=("amount_usd", "sum"),
        confirmed_gmv=("amount_usd", lambda x: x[transactions.loc[x.index, "status_clean"] == "captured"].sum()),
        avg_risk_score=("risk_score", "mean"),
        high_value_transactions=("high_value_flag", "sum"),
        high_risk_transactions=("high_risk_flag", "sum")
    )
    .reset_index()
)

merchant_performance_summary["avg_risk_score"] = merchant_performance_summary["avg_risk_score"].round(2)

merchant_performance_summary.to_csv("/content/processed/merchant_performance_summary.csv", index=False)

display(merchant_performance_summary)

,merchant_name,total_transactions,total_gmv,confirmed_gmv,avg_risk_score,high_value_transactions,high_risk_transactions
0,ALPHA MART,11,40698.0,29928.5,61.20,2,3
1,BETA STORES,11,41531.0,33141.5,69.36,2,5
2,City Pharma,2,8640.0,8640.0,40.00,0,0
3,DELTA TRAVELS,4,14600.0,10300.0,48.75,1,0
4,Eco Home,2,10152.0,0.0,54.50,0,0
